# Step 6 — PatchTST v9 (v8.1 architecture, v7 sliding-window data)

**This runs the v8.1 architecture on the new sliding-window dataset.**
No congestion head. That is deliberate: the data changed, so changing the
architecture at the same time would make the result unattributable. Run this
against the 0.316 baseline first; v8.2's congestion head is a separate
experiment afterwards (appendix at the bottom).

---

## What changed from v8.2, and why

| # | Change | Why |
|---|---|---|
| 1 | `STEPS_IN/OUT/DT_MS` read from `feature_schema.json` | v8.2 hardcoded `2500, 500, 10`. On 1500-step data the Dataset builds a 2500-row `extra` array that will not concat with the 1500-row scaled input. |
| 2 | `patch_size` chosen by `pick_patch_size()` | **`patch_size=50` on 1500 inputs is a crash.** 1500//50 = 30 patches, 500//30 = 16, 30x16 = **480 != 500**. `pred_rtt` comes out 480 long against a 500-long `y_rtt`. Correct value is **30**, which gives 50 patches — the same transformer sequence length v8.1 had. |
| 3 | Scaler **fingerprint** check | v8.2 asserted `rtt mean == 80.2735`, i.e. it checked the scalers were the OLD ones. `scalers.json` sits beside the new dataset, so a stale file loads **silently**. Run `step5_fit_scalers.py` first. |
| 4 | `num_workers=4`, `persistent_workers`, `prefetch_factor` | ~2 ms/sample of npz decompress + scaling. At `num_workers=0` that is ~8 min/epoch of loading with the GPU idle. |
| 5 | `eval_batch=128` | val/test do no backward pass; batch 32 wasted most of the GPU. |
| 6 | `CUDA_LAUNCH_BLOCKING` removed | Debugging flag left on from a past session. It serializes every CUDA call. |
| 7 | Throughput probe cell | Times 30 real steps and projects total hours **before** you commit to the run. |
| 8 | Shape asserts use the schema | Not a hardcoded 2500. |

## Order

1. `python step5_fit_scalers.py --root <dataset root>`  &larr; **do this first**
2. Run this notebook top to bottom.
3. Stop at the throughput probe and read the projection.


## Cell 1 — Imports

In [ ]:
import json, warnings, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# CUDA_LAUNCH_BLOCKING was set to '1' in v8.2. That is a DEBUGGING flag: it
# makes every CUDA call synchronous so stack traces point at the right kernel.
# It also throws away all kernel-launch overlap. With 6x the data that is a
# real cost for no benefit. Turn it on only while chasing a device-side assert.
os.environ.pop('CUDA_LAUNCH_BLOCKING', None)
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False

print('ON_COLAB:', ON_COLAB)
print('torch:', torch.__version__)
print('cuda :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ON_COLAB: True
torch: 2.11.0+cu130
cuda : True NVIDIA L4


## Cell 2 — Colab: extract dataset

In [ ]:
import time
import shutil
from pathlib import Path

EXTRACT_TO = Path('/content')

# Drive paths
TAR_PATH_DRIVE = Path('/content/drive/MyDrive/rtt_ds_1s_samples.tar')
ZIP_PATH_DRIVE = Path('/content/drive/MyDrive/rtt_ds_1s_samples.zip')

# Local paths for the temporary copied archives
LOCAL_TAR = EXTRACT_TO / TAR_PATH_DRIVE.name
LOCAL_ZIP = EXTRACT_TO / ZIP_PATH_DRIVE.name

target = EXTRACT_TO / 'rtt_ds_1s_samples'

if target.exists():
    print('Already extracted.')
elif TAR_PATH_DRIVE.exists():
    t0 = time.time()

    # Step 1: Copy to Colab's local NVMe storage
    print(f'Copying {TAR_PATH_DRIVE.name} to local storage...')
    shutil.copy2(TAR_PATH_DRIVE, LOCAL_TAR)
    print(f'Copy done in {(time.time()-t0)/60:.1f} min. Extracting...')

    # Step 2: Extract locally
    t1 = time.time()
    !tar -xf "{LOCAL_TAR}" -C "{EXTRACT_TO}"
    print(f'Extraction done in {(time.time()-t1)/60:.1f} min')

    # Step 3: Clean up the local archive
    LOCAL_TAR.unlink()
    print(f'Total time: {(time.time()-t0)/60:.1f} min')

elif ZIP_PATH_DRIVE.exists():
    t0 = time.time()

    # Step 1: Copy to Colab's local NVMe storage
    print(f'Copying {ZIP_PATH_DRIVE.name} to local storage...')
    shutil.copy2(ZIP_PATH_DRIVE, LOCAL_ZIP)
    print(f'Copy done in {(time.time()-t0)/60:.1f} min. Extracting...')

    # Step 2: Extract locally
    t1 = time.time()
    !unzip -q "{LOCAL_ZIP}" -d "{EXTRACT_TO}"
    print(f'Extraction done in {(time.time()-t1)/60:.1f} min')

    # Step 3: Clean up the local archive
    LOCAL_ZIP.unlink()
    print(f'Total time: {(time.time()-t0)/60:.1f} min')

else:
    raise FileNotFoundError('No rtt_forecast_ds_1s_samples.tar or .zip found on Drive')

# Count extracted files
# Copy non-zipped files from Drive


n = sum(1 for _ in (target / 'samples').rglob('*.npz'))
print(f'{n:,} .npz files present')

Copying rtt_ds_1s_samples.zip to local storage...


OSError: [Errno 28] No space left on device: '/content/drive/MyDrive/rtt_ds_1s_samples.zip' -> '/content/rtt_ds_1s_samples.zip'

## Cell 3 — Paths

In [ ]:
if ON_COLAB:
    LOCAL_ROOT      = Path('/content/rtt_forecast_ds_1s')
    DRIVE_ROOT      = Path('/content/drive/MyDrive/rtt_forecast_ds_1s')
    CKPT_BASE       = LOCAL_ROOT / 'checkpoints'
    DRIVE_CKPT_BASE = DRIVE_ROOT / 'checkpoints'
else:
    LOCAL_ROOT = Path(r'D:\LENS Dataset\combined\rtt_forecast_ds')
    CKPT_BASE  = LOCAL_ROOT / 'checkpoints'

SAMPLE_DIR = LOCAL_ROOT / 'samples'
CKPT_DIR   = CKPT_BASE / 'patchtst_v9.13'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if ON_COLAB:
    DRIVE_CKPT_DIR = DRIVE_CKPT_BASE / 'patchtst_v9.13'
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print('LOCAL_ROOT :', LOCAL_ROOT)
print('CKPT_DIR   :', CKPT_DIR)

In [ ]:
from pathlib import Path

target = Path('/content/rtt_forecast_ds_1s')
_drive_ds = Path('/content/drive/MyDrive/rtt_forecast_ds_1s')

print(f"target exists: {target.exists()}")
print(f"drive_ds exists: {_drive_ds.exists()}")

if _drive_ds.exists():
    print(f"Drive contents: {[f.name for f in _drive_ds.iterdir()]}")
else:
    print("Drive folder not found. Check path.")
    # Try listing what's in MyDrive
    import os
    drive = Path('/content/drive/MyDrive')
    matches = [f for f in os.listdir(drive) if 'forecast' in f.lower() or '1s' in f.lower()]
    print(f"Similar names in Drive: {matches}")

In [ ]:
target = Path('/content/rtt_forecast_ds_1s')
for f in ['index_split_roccet_05s.csv', 'scalers.json', 'feature_schema.json']:
    print(f"{f}: {(target / f).exists()}")

In [ ]:
import shutil
from pathlib import Path

drive_source_dir = Path('/content/drive/MyDrive/rtt_forecast_ds_1s')
local_target_dir = Path('/content/rtt_forecast_ds_1s')

local_target_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    'index_split_roccet_05s.csv',
    'scalers.json',
    'feature_schema.json'
]

print(f"Copying files from {drive_source_dir} to {local_target_dir}:")
for filename in files_to_copy:
    source_path = drive_source_dir / filename
    destination_path = local_target_dir / filename
    if source_path.exists():
        shutil.copy2(source_path, destination_path)
        print(f"  - Copied: {filename}")
    else:
        print(f"  - Warning: Source file not found: {filename}")

# Verify files are now present in the target directory
print("\nVerifying files in local directory:")
for f in files_to_copy:
    print(f"{f}: {(local_target_dir / f).exists()}")

## Cell 4 — Schema, scalers, patch size  `[REWRITTEN]`

In [ ]:
# =============================================================================
# Schema, scalers, patch size.  EVERYTHING geometric is read from the schema —
# nothing about the window length is hardcoded in this notebook any more.
# =============================================================================
LOCAL_ROOT      = Path('/content/rtt_forecast_ds_1s')
SAMPLE_DIR = Path('/content/samples')
with open(LOCAL_ROOT / 'feature_schema.json') as f:
    SCHEMA = json.load(f)

STEPS_IN  = int(SCHEMA['steps_in'])      # 1500 for the v7 sliding dataset
STEPS_OUT = 100
DT_MS     = int(SCHEMA['dt_ms'])         # 10

SELECT_IDX   = SCHEMA['x_num_select_indices']
NUM_FEATURES = SCHEMA['num_features']
F_NUM_BASE   = len(NUM_FEATURES)
F_NUM        = F_NUM_BASE + 2            # + time_since_ho_norm, ho_time_known

print(f"dataset  : {SCHEMA.get('version')}  windowing={SCHEMA.get('windowing')}")
print(f"geometry : steps_in={STEPS_IN}  steps_out={STEPS_OUT}  dt={DT_MS}ms")
if SCHEMA.get('window_s'):
    print(f"           window={SCHEMA['window_s']}s  horizon={SCHEMA['horizon_s']}s"
          f"  step={SCHEMA['step_s']}s")

# ── PATCH SIZE ───────────────────────────────────────────────────────────────
# The decoder does  x.repeat_interleave(steps_per_token)  over n_patches, where
#     n_patches       = STEPS_IN  // patch_size
#     steps_per_token = STEPS_OUT // n_patches
# so the output length is n_patches * steps_per_token and that MUST equal
# STEPS_OUT. With STEPS_IN=1500 and patch_size=50 you get 30 patches,
# 500//30 = 16, and 30*16 = 480 != 500 -> pred_rtt is 480 long while y_rtt is
# 500 and compute_loss dies. Pick a size that divides cleanly instead.
def pick_patch_size(steps_in, steps_out, prefer_n_patches=50):
    ok = [p for p in range(2, steps_in + 1)
          if steps_in % p == 0 and steps_out % (steps_in // p) == 0]
    if not ok:
        raise ValueError(f'no valid patch_size for {steps_in}/{steps_out}')
    return min(ok, key=lambda p: abs(steps_in // p - prefer_n_patches))

PATCH_SIZE      = pick_patch_size(STEPS_IN, STEPS_OUT, prefer_n_patches=50)
N_PATCHES       = STEPS_IN // PATCH_SIZE
STEPS_PER_TOKEN = STEPS_OUT // N_PATCHES
assert N_PATCHES * STEPS_PER_TOKEN == STEPS_OUT
print(f"patching : patch_size={PATCH_SIZE}  n_patches={N_PATCHES}  "
      f"steps_per_token={STEPS_PER_TOKEN}  -> decoder emits "
      f"{N_PATCHES*STEPS_PER_TOKEN} == STEPS_OUT  OK")

# ── SCALERS + FINGERPRINT ────────────────────────────────────────────────────
# v8.2 asserted rtt mean == 80.2735, i.e. it checked that the scalers were the
# OLD ones. scalers.json lives in the same folder as the new dataset, so a
# stale file loads silently and you train on normalization fitted to 25s
# windows. Verify the fingerprint written by step5_fit_scalers.py instead.
with open(target / 'scalers.json') as f:
    _RAW_SCALERS = json.load(f)

_FP = _RAW_SCALERS.get('__fingerprint__')
if _FP is None:
    raise RuntimeError(
        "scalers.json has no __fingerprint__ block. It was written by the OLD "
        "Step 5 and is almost certainly fitted to 25-second windows. "
        "Re-run:  python step5_fit_scalers.py --root <dataset root>")

_mismatch = [f"{k}: scalers={_FP.get(k)!r} vs schema={SCHEMA.get(k)!r}"
             for k in ('steps_in', 'steps_out', 'window_s', 'step_s')
             if _FP.get(k) != SCHEMA.get(k)]
if _mismatch:
    raise RuntimeError("STALE SCALERS — fingerprint does not match this dataset:\n  "
                       + "\n  ".join(_mismatch)
                       + "\nRe-run step5_fit_scalers.py.")
if list(_FP.get('num_features', [])) != list(NUM_FEATURES):
    raise RuntimeError("Scalers were fitted on a different feature list.")

SCALERS = {k: v for k, v in _RAW_SCALERS.items() if k in NUM_FEATURES}
assert set(NUM_FEATURES) == set(SCALERS), \
    f"Missing scalers: {set(NUM_FEATURES) - set(SCALERS)}"

_p = SCALERS['rtt_ms']['params']
print(f"\nscalers  : fingerprint OK  (fitted on {_FP['n_fit_windows']:,} of "
      f"{_FP['n_train_windows']:,} train windows)")
print(f"           rtt_ms clip={_p['clip_max']:.0f} mean={_p['mean']:.4f} "
      f"scale={_p['scale']:.4f}")
print(f"           v6 reference was mean=80.2735 scale=47.91 on 25s windows — "
      f"a DIFFERENT number here is expected and correct.")

# ── categorical vocab ────────────────────────────────────────────────────────
CAT_FEATURES, CAT_COL_IDX = ['satellite_id_id'], 0
_idx = pd.read_csv(LOCAL_ROOT / 'index_split_roccet_05s.csv')

_sat_card = None
for _k in ('cat_vocab_sizes', 'x_cat_vocab_sizes', 'cat_cardinalities',
           'satellite_id_vocab', 'sat_vocab_size', 'n_satellites'):
    if _k in SCHEMA:
        _v = SCHEMA[_k]
        _sat_card = int(_v[CAT_COL_IDX]) if isinstance(_v, (list, tuple)) else int(_v)
        break

_N_SCAN = 4000
_rng_v  = np.random.default_rng(0)
_scan_i = _rng_v.choice(len(_idx), size=min(_N_SCAN, len(_idx)), replace=False)
_seen = set()
for _j in _scan_i:
    _d = np.load(SAMPLE_DIR / _idx['sample'].iloc[int(_j)], allow_pickle=False)
    _seen.update(np.unique(_d['X_cat'][:, CAT_COL_IDX]).tolist())
_scan_max = int(max(_seen))
print(f"\nsat scan : {len(_seen)} distinct IDs, max={_scan_max} "
      f"(from {len(_scan_i):,} random windows)")

if _sat_card is None:
    SAT_VOCAB_SIZE = _scan_max + 200
    print(f"           no vocab in schema -> scan max + 200 = {SAT_VOCAB_SIZE}")
else:
    SAT_VOCAB_SIZE = max(_sat_card, _scan_max + 1)
CAT_VOCAB_SIZES = [SAT_VOCAB_SIZE]
print(f"           SAT_VOCAB_SIZE = {SAT_VOCAB_SIZE}")

SAT_CHANGE_IDX = NUM_FEATURES.index('sat_change')
SINR_IDX       = NUM_FEATURES.index('AVERAGE_SINR')
SINR_DELTA_IDX = NUM_FEATURES.index('sinr_delta')
SPIKE_THRESH   = float(np.log1p(300.0))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"\nF_NUM_BASE={F_NUM_BASE}  F_NUM={F_NUM}  windows={len(_idx):,}")
print(f"Device: {DEVICE}")


## Cell 5 — apply_scalers  *(unchanged from v8.2)*

In [ ]:
FFILL_COLS = ('rtt_ms',)

def _ffill_1d(a: np.ndarray) -> np.ndarray:
    """Forward-fill NaNs; back-fill leading NaNs. All-NaN returned unchanged."""
    a = a.astype(np.float32, copy=True)
    fin = np.isfinite(a)
    if not fin.any():
        return a
    idx = np.where(fin, np.arange(len(a)), 0)
    np.maximum.accumulate(idx, out=idx)
    out = a[idx]
    first = int(np.argmax(fin))
    if first > 0:
        out[:first] = a[first]
    return out


def apply_scalers(X_num: np.ndarray) -> np.ndarray:
    """Apply fitted scalers to a (N, 9) column-selected X_num.
    MUST stay identical to Step 5's apply_scalers."""
    out = X_num.copy().astype(np.float32)

    for i, col in enumerate(NUM_FEATURES):          # ffill BEFORE scaling
        if col in FFILL_COLS:
            out[:, i] = _ffill_1d(out[:, i])

    for i, col in enumerate(NUM_FEATURES):
        if col not in SCALERS:
            continue
        info, p = SCALERS[col], SCALERS[col]['params']
        valid = np.isfinite(out[:, i])
        if not valid.any():
            continue
        t = info['type']
        if t == 'clipped_standard':
            v = np.clip(out[valid, i], 0.0, p['clip_max'])
            out[valid, i] = (v - p['mean']) / p['scale']
        elif t == 'standard':
            out[valid, i] = (out[valid, i] - p['mean']) / p['scale']
        elif t == 'log1p_standard':
            v = np.clip(out[valid, i], 0, p['clip_max'])
            out[valid, i] = (np.log1p(v) - p['mean']) / p['scale']
        elif t == 'robust':
            out[valid, i] = (out[valid, i] - p['center']) / p['scale']
    return np.nan_to_num(out, nan=0.0)

print('apply_scalers (with ffill) defined OK')

## Cell 6 — Dataset  *(unchanged code; STEPS_IN now comes from the schema)*

In [ ]:
class RTTForecastDataset(Dataset):
    def __init__(self, index_df: pd.DataFrame):
        self.samples = index_df['sample'].tolist()
        ho = index_df['horizon_handover'].values.astype(np.float32)
        cong = ((index_df['event_class'].values == 2).astype(np.float32)
                if 'event_class' in index_df.columns
                else np.zeros(len(index_df), dtype=np.float32))
        self.event_label = np.clip(ho + cong, 0.0, 1.0).astype(np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = np.load(SAMPLE_DIR / self.samples[idx], allow_pickle=False)

        X_num_raw = d['X_num'][:, SELECT_IDX].copy()  # (2500, 9)

        base_s   = float(d['time_since_ho'])
        ho_known = float(d['ho_time_known'])

        sat_col  = X_num_raw[:, SAT_CHANGE_IDX]
        sat_bins = np.where(sat_col > 0.5)[0]
        indices  = np.arange(STEPS_IN, dtype=np.float64)

        if len(sat_bins) == 0:
            tsb = (base_s + (indices + 1) * (DT_MS / 1000.0)).astype(np.float32)
            tsn = np.clip(tsb / 150.0, 0.0, 2.0)
            hkc = np.full(STEPS_IN, ho_known, dtype=np.float32)
        else:
            counts    = np.searchsorted(sat_bins, np.arange(STEPS_IN), side='right')
            has_prior = counts > 0
            last_ho   = sat_bins[np.clip(counts - 1, 0, len(sat_bins) - 1)]
            tsb = np.where(
                has_prior,
                (indices - last_ho) * (DT_MS / 1000.0),
                base_s + (indices + 1) * (DT_MS / 1000.0),
            ).astype(np.float32)
            tsn = np.clip(tsb / 150.0, 0.0, 2.0)
            hkc = np.where(has_prior, 1.0, ho_known).astype(np.float32)

        extra = np.stack([tsn, hkc], axis=1)
        SINR_DELTA_WINDOW = 100        # 100 bins x 10 ms = 1 s

        _scaled = apply_scalers(X_num_raw)
        W = SINR_DELTA_WINDOW
        _scaled[:, SINR_DELTA_IDX] = np.concatenate([
        np.zeros(W, np.float32),
        _scaled[W:, SINR_IDX] - _scaled[:-W, SINR_IDX],
        ]).astype(np.float32)
        X_num = np.concatenate([_scaled, extra], axis=1)

        X_cat = d['X_cat'][:, CAT_COL_IDX:CAT_COL_IDX+1].astype(np.int64)
        X_cat = np.clip(X_cat, 0, SAT_VOCAB_SIZE)

        y_rtt    = d['y_rtt'][:STEPS_OUT].astype(np.float32)
        y_loss   = d['y_loss'][:STEPS_OUT].astype(np.float32)
        mask     = d['rtt_mask'][:STEPS_OUT].astype(np.float32)
        spike_lb = (np.nan_to_num(y_rtt, nan=0.0) > SPIKE_THRESH).astype(np.float32)
        ev_lb    = np.float32(self.event_label[idx])

        return (torch.from_numpy(X_num), torch.from_numpy(X_cat),
                torch.from_numpy(y_rtt), torch.from_numpy(y_loss),
                torch.from_numpy(mask), torch.from_numpy(spike_lb),
                torch.tensor(ev_lb))

print('RTTForecastDataset defined OK')

## Cell 7 — DataLoaders  `[REWRITTEN]`

In [ ]:
EVENT_POS_WEIGHT = 19.0     # placeholder; get_loaders() overwrites this

# ── Data-loading throughput is the bottleneck, not the GPU ───────────────────
# Each __getitem__ does: open .npz -> zlib decompress -> ffill -> scale.
# That is ~2 ms single-threaded. The v7 train split after keep_ratio=0.25 is
# ~234k windows, so num_workers=0 costs ~8 min/epoch of pure loading with the
# GPU idle. On Colab (Linux, fork) workers are free money.
#
# On local WINDOWS, num_workers>0 spawns processes that re-import the module
# and cannot see a Dataset class defined in a notebook cell — same trap as
# Step 4. Move RTTForecastDataset into a .py file there, or use 0.
NUM_WORKERS   = 8 if ON_COLAB else 0
EVAL_BATCH    = 128     # no gradients in val/test, so use a much bigger batch
PREFETCH      = 4


def _event_label_vec(df: pd.DataFrame) -> np.ndarray:
    """handover OR congestion — must match RTTForecastDataset.event_label."""
    ho = df['horizon_handover'].values.astype(np.float32)
    cong = ((df['event_class'].values == 2).astype(np.float32)
            if 'event_class' in df.columns
            else np.zeros(len(df), dtype=np.float32))
    return np.clip(ho + cong, 0.0, 1.0)



def get_loaders(batch_size=128, num_workers=0,
                normal_keep_ratio=0.05,      # <-- was 0.15
                ho_keep_ratio=1.0,           # <-- NEW: downsample handover too
                seed=42):
    global EVENT_POS_WEIGHT
    index = pd.read_csv(target / 'index_split_roccet_05s.csv')

    tr = index[index.split == 'train'].reset_index(drop=True)
    va = index[index.split == 'val'].reset_index(drop=True)
    te = index[index.split == 'test'].reset_index(drop=True)

    rng = np.random.default_rng(seed)

    # ---- downsample NORMAL (class 0) ----
    is_norm = (tr.event_class == 0).values
    keep_norm = ~is_norm | (rng.random(len(tr)) < normal_keep_ratio)

    # ---- downsample HANDOVER (class 1) — NEW ----
    is_ho = (tr.event_class == 1).values
    keep_ho = ~is_ho | (rng.random(len(tr)) < ho_keep_ratio)

    # keep a row if it survives BOTH filters (congestion always kept: both True)
    keep = keep_norm & keep_ho
    tr = tr[keep].reset_index(drop=True)

    # ---- report the resulting mix ----
    n0 = int((tr.event_class == 0).sum())
    n1 = int((tr.event_class == 1).sum())
    n2 = int((tr.event_class == 2).sum())
    ntot = len(tr)
    print(f"Train downsampled -> {ntot:,}  "
          f"(normal={n0:,} {n0/ntot:.1%}  "
          f"HO={n1:,} {n1/ntot:.1%}  "
          f"cong={n2:,} {n2/ntot:.1%})")
    print(f"  normal_keep={normal_keep_ratio}  ho_keep={ho_keep_ratio}")
    print(f"  HO:cong ratio = {n1/max(n2,1):.1f}:1  (was ~9:1 before HO downsample)")

    # ---- event pos_weight from the NEW mix ----
    ev = ((tr.event_class == 1) | (tr.event_class == 2)).values
    n_ev = int(ev.sum()); n_neg = int((~ev).sum())
    EVENT_POS_WEIGHT = n_neg / max(n_ev, 1)
    print(f"  EVENT_POS_WEIGHT = {EVENT_POS_WEIGHT:.2f}  "
          f"(event prevalence in train = {n_ev/ntot:.3f})")
    print(f"  NOTE: val/test kept at TRUE prevalence "
          f"(val event={((va.event_class!=0).mean()):.3f} "
          f"test event={((te.event_class!=0).mean()):.3f})")

    pin = (DEVICE.type == 'cuda')
    kw = dict(num_workers=num_workers, pin_memory=pin)
    return (
        DataLoader(RTTForecastDataset(tr), batch_size=batch_size, shuffle=True, **kw),
        DataLoader(RTTForecastDataset(va), batch_size=batch_size, shuffle=False, **kw),
        DataLoader(RTTForecastDataset(te), batch_size=batch_size, shuffle=False, **kw),
    )

print('get_loaders defined OK')


## Cell 8 — Loss  *(unchanged from v8.1)*

In [ ]:
_huber = nn.HuberLoss(delta=1.0, reduction='none')

# Weight applied to horizon bins where RTT > 300ms in the regression loss.
# Was 0.1 (10x suppression). Huber already handles the heavy tail, so 0.1 was
# double-suppressing exactly the events Stage 2 needs to see.
# 1.0 = no suppression, 0.5 = mild. Ablate 0.5 vs 1.0 (see handover 6.1).
SPIKE_RTT_WEIGHT = 0.5

# Per-bin spike head. Left at the historical value; unlike the event head this
# class is genuinely rare and is NOT affected by the sample-level downsampling.
SPIKE_POS_WEIGHT = 25.0


def compute_loss(pred_rtt, pred_loss, pred_spike, pred_event,
                 y_rtt, y_loss, mask, spike_labels, event_labels):
    valid = (mask > 0) & torch.isfinite(y_rtt)

    rtt_w = torch.ones_like(y_rtt)
    rtt_w[spike_labels > 0.5] = SPIKE_RTT_WEIGHT
    l_r = (_huber(pred_rtt[valid], y_rtt[valid]) * rtt_w[valid]).mean()

    l_spike = F.binary_cross_entropy_with_logits(
        pred_spike[valid], spike_labels[valid],
        pos_weight=torch.tensor(SPIKE_POS_WEIGHT, device=pred_spike.device))

    # EVENT_POS_WEIGHT is resolved at call time -> always matches get_loaders()
    l_event = F.binary_cross_entropy_with_logits(
        pred_event, event_labels.float(),
        pos_weight=torch.tensor(float(EVENT_POS_WEIGHT),
                                device=pred_event.device))

    vl = torch.isfinite(y_loss) & (mask > 0)
    l_loss = (F.binary_cross_entropy_with_logits(pred_loss[vl], y_loss[vl])
              if vl.sum() > 0
              else torch.tensor(0.0, device=pred_rtt.device))

    loss = l_r + 0.3 * l_loss + 0.1 * l_spike + 0.3 * l_event
    return loss, l_r.item(), l_loss.item(), l_spike.item(), l_event.item()

print(f'compute_loss defined OK  (SPIKE_RTT_WEIGHT={SPIKE_RTT_WEIGHT}, '
      f'SPIKE_POS_WEIGHT={SPIKE_POS_WEIGHT}, '
      f'EVENT_POS_WEIGHT set by get_loaders)')

## Cell 9 — Model  `[REWRITTEN: patch_size guard]`

In [ ]:
class PatchTSTForecaster(nn.Module):
    """v8.1 architecture — NO congestion head.

    This is deliberate. The v7 dataset changed; adding the v8.2 congestion head
    at the same time would change two things at once and you could not
    attribute the result. Run this first against the 0.316 baseline, then
    v8.2 as a separate experiment.
    """
    def __init__(s, f_num=11, cat_vocab_sizes=None, emb_dim=8,
                 patch_size=None, d_model=256, n_heads=8, n_layers=4,
                 d_ff=512, dropout=0.1, spike_dropout=0.1):
        super().__init__()
        cat_vocab_sizes = cat_vocab_sizes or []
        s.patch_size      = PATCH_SIZE if patch_size is None else patch_size
        s.n_patches       = STEPS_IN // s.patch_size
        s.steps_per_token = STEPS_OUT // s.n_patches

        # Guard the exact bug that patch_size=50 on 1500 inputs produces.
        assert s.n_patches * s.patch_size == STEPS_IN, (
            f'patch_size {s.patch_size} does not divide STEPS_IN {STEPS_IN}')
        assert s.n_patches * s.steps_per_token == STEPS_OUT, (
            f'decoder would emit {s.n_patches * s.steps_per_token} steps but '
            f'STEPS_OUT is {STEPS_OUT}. patch_size={s.patch_size} gives '
            f'{s.n_patches} patches and {STEPS_OUT}//{s.n_patches}='
            f'{s.steps_per_token}. Use pick_patch_size().')

        s.embeddings = nn.ModuleList([nn.Embedding(v + 1, emb_dim, padding_idx=0)
                                      for v in cat_vocab_sizes])
        ft = f_num + emb_dim * len(cat_vocab_sizes)
        s.input_pad = (-ft) % 8
        sd = 32
        s.ts_proj = nn.Sequential(nn.Linear(ft + s.input_pad, sd), nn.GELU())
        s.patch_embed = nn.Sequential(
            nn.Linear(s.patch_size * sd, d_model), nn.LayerNorm(d_model))
        s.pos_embed = nn.Parameter(torch.zeros(1, s.n_patches, d_model))
        nn.init.trunc_normal_(s.pos_embed, std=0.02)
        el = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout,
                                        batch_first=True, norm_first=True)
        s.transformer = nn.TransformerEncoder(el, n_layers)
        s.drop = nn.Dropout(dropout)
        s.rtt_decoder = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                      nn.Dropout(dropout), nn.Linear(d_model, 1))
        s.loss_decoder = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                       nn.Dropout(dropout), nn.Linear(d_model, 1))
        s.spike_decoder = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.GELU(),
                                        nn.Dropout(spike_dropout),
                                        nn.Linear(d_model // 2, 1))
        s.event_head = nn.Sequential(nn.Linear(d_model, d_model // 4), nn.GELU(),
                                     nn.Dropout(dropout), nn.Linear(d_model // 4, 1))

    def forward(s, Xn, Xc):
        B = Xn.shape[0]
        if s.embeddings:
            e = torch.cat([emb(Xc[:, :, i].clamp(0, emb.num_embeddings - 1))
                           for i, emb in enumerate(s.embeddings)], -1)
            x = torch.cat([Xn, e], -1)
        else:
            x = Xn
        if s.input_pad:
            x = F.pad(x, (0, s.input_pad))
        x = s.ts_proj(x)
        x = x.reshape(B, s.n_patches, s.patch_size * x.shape[-1])
        x = s.patch_embed(x) + s.pos_embed
        x = s.transformer(s.drop(x))
        pooled = x.mean(dim=1)
        pred_event = s.event_head(pooled).squeeze(-1)
        xr = s.drop(x.repeat_interleave(s.steps_per_token, dim=1))
        return (s.rtt_decoder(xr).squeeze(-1),
                s.loss_decoder(xr).squeeze(-1),
                s.spike_decoder(xr).squeeze(-1),
                pred_event)

print(f'PatchTSTForecaster (v8.1 arch) defined  patch_size={PATCH_SIZE} '
      f'n_patches={N_PATCHES} steps_per_token={STEPS_PER_TOKEN}')


## Cell 10 — Instantiate + shape check  `[REWRITTEN]`

In [ ]:
import pandas as pd
idx = pd.read_csv("/content/rtt_forecast_ds_1s/index_split_roccet_05s.csv", nrows=0)
print('horizon_handover' in idx.columns)
print(len(idx.columns))

In [ ]:
model = PatchTSTForecaster(
    f_num=F_NUM, cat_vocab_sizes=CAT_VOCAB_SIZES,
    emb_dim=8, patch_size=PATCH_SIZE,
    d_model=256, n_heads=8, n_layers=4, d_ff=512,
    dropout=0.1, spike_dropout=0.3,
).to(DEVICE)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
f_total  = F_NUM + 8 * len(CAT_VOCAB_SIZES)
f_padded = f_total + ((-f_total) % 8)
print(f'f_total={f_total} -> f_padded={f_padded} -> ts_proj->32 -> '
      f'patch_dim={PATCH_SIZE*32}')

# Shape check against the SCHEMA, not a hardcoded 2500.
_ds = RTTForecastDataset(pd.read_csv(target / 'index_split_roccet_05s.csv').head(2))
_xn, _xc, _yr, _yl, _m, _spk, _ev = _ds[0]
print(f'X_num: {tuple(_xn.shape)}  expect ({STEPS_IN}, {F_NUM})')
print(f'X_cat: {tuple(_xc.shape)}  expect ({STEPS_IN}, 1)')
print(f'y_rtt: {tuple(_yr.shape)}  expect ({STEPS_OUT},)')
assert _xn.shape == (STEPS_IN, F_NUM), f'X_num shape {_xn.shape}'
assert _xc.shape == (STEPS_IN, 1),     f'X_cat shape {_xc.shape}'
assert _yr.shape == (STEPS_OUT,),      f'y_rtt shape {_yr.shape}'

model.eval()
with torch.no_grad():
    _pr, _pl, _ps, _pe = model(_xn.unsqueeze(0).to(DEVICE),
                               _xc.unsqueeze(0).to(DEVICE))
print(f'pred_rtt: {tuple(_pr.shape)}  expect (1, {STEPS_OUT})')
assert _pr.shape == (1, STEPS_OUT), (
    f'DECODER LENGTH MISMATCH: {_pr.shape[1]} != {STEPS_OUT}. '
    f'This is the patch_size bug — see pick_patch_size().')
model.train()
print('Shape check PASSED')


## Cell 11 — Training config

> Builds the loaders and the optimizer, so it must run AFTER the model exists.

In [ ]:
EPOCHS       = 20
BATCH_SIZE   = 128
LR_MAX       = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 12
NORMAL_KEEP_RATIO = 0.05

# Which validation metric drives early stopping and best.pt.
# 'ev_prauc' = the stated primary objective (event classification).
# 'val_loss' = total multi-task loss, dominated by the RTT regression term.
EARLY_STOP_ON = 'ev_prauc'
assert EARLY_STOP_ON in ('ev_prauc', 'val_loss')

train_dl, val_dl, test_dl = get_loaders(
    batch_size=BATCH_SIZE, num_workers=8,
    normal_keep_ratio=NORMAL_KEEP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX,
                              weight_decay=WEIGHT_DECAY)
steps_per_epoch = len(train_dl)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_MAX, epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.10, anneal_strategy='cos',
    div_factor=25, final_div_factor=1000)

ckpt_best       = CKPT_DIR / 'best.pt'
ckpt_best_event = CKPT_DIR / 'best_event.pt'
ckpt_last       = CKPT_DIR / 'last.pt'

best_val         = float('inf')
best_event_prauc = 0.0
start_ep         = 1
patience_ctr     = 0

if ckpt_last.exists():
    state = torch.load(ckpt_last, map_location=DEVICE)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optim'])
    if 'sched' in state:
        scheduler.load_state_dict(state['sched'])
    else:
        print('  WARNING: checkpoint has no scheduler state — LR schedule '
              'restarts from warmup. Expect a loss bump.')
    best_val         = state.get('best_val', float('inf'))
    best_event_prauc = state.get('best_event_prauc', 0.0)
    patience_ctr     = state.get('patience_ctr', 0)
    start_ep         = state.get('epoch', 0) + 1
    print(f'Resumed from ep {start_ep-1}  best_val={best_val:.4f}  '
          f'best_event_prauc={best_event_prauc:.4f}')
else:
    print(f'Fresh start | lr={LR_MAX} wd={WEIGHT_DECAY} patience={PATIENCE}')
    print(f'  normal_keep_ratio={NORMAL_KEEP_RATIO}  '
          f'event_pos_weight={EVENT_POS_WEIGHT:.2f}  '
          f'spike_rtt_weight={SPIKE_RTT_WEIGHT}')
    print(f'  early stopping on: {EARLY_STOP_ON}')
    print(f'Steps/epoch: {steps_per_epoch}')

## Cell 12 — Throughput probe  `[NEW — read the projection]`

In [ ]:
# =============================================================================
# THROUGHPUT PROBE — run this BEFORE committing to 40 epochs.
# Times a few real training steps and extrapolates. If the projection is
# unacceptable, change NUM_WORKERS / BATCH_SIZE now, not 3 hours in.
# =============================================================================
import itertools

PROBE_STEPS = 30
model.train()
_opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

# warm up (first batches include worker startup and cuDNN autotune)
for batch in itertools.islice(train_dl, 3):
    xn, xc, yr, yl, msk, spk, ev = [b.to(DEVICE) for b in batch]
    _opt.zero_grad()
    pr, pl, ps, pe = model(xn, xc)
    loss, *_ = compute_loss(pr, pl, ps, pe, yr, yl, msk, spk, ev)
    loss.backward(); _opt.step()
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

t0 = time.time()
n = 0
for batch in itertools.islice(train_dl, PROBE_STEPS):
    xn, xc, yr, yl, msk, spk, ev = [b.to(DEVICE) for b in batch]
    _opt.zero_grad()
    pr, pl, ps, pe = model(xn, xc)
    loss, *_ = compute_loss(pr, pl, ps, pe, yr, yl, msk, spk, ev)
    loss.backward(); _opt.step(); n += 1
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
sec_per_step = (time.time() - t0) / max(n, 1)

steps = len(train_dl)
val_steps = len(val_dl)
train_min = steps * sec_per_step / 60
val_min   = val_steps * sec_per_step * 0.35 / 60   # eval: no backward, bigger batch

print(f'  {sec_per_step*1000:.0f} ms / train step  ({BATCH_SIZE} samples)')
print(f'  {steps:,} train steps -> ~{train_min:.1f} min/epoch')
print(f'  {val_steps:,} val steps -> ~{val_min:.1f} min/epoch (rough)')
print(f'  ~{train_min+val_min:.1f} min/epoch  ->  {EPOCHS} epochs = '
      f'{(train_min+val_min)*EPOCHS/60:.1f} HOURS')
print()
if (train_min + val_min) * EPOCHS / 60 > 8:
    print('  >>> Over 8 hours. Colab will very likely disconnect first.')
    print('  >>> Options: raise NUM_WORKERS, raise BATCH_SIZE (64/128),')
    print('  >>>          lower EPOCHS, or lower normal_keep_ratio.')
    print('  >>> Checkpointing to Drive means a disconnect is recoverable,')
    print('  >>> but you will lose the rest of the session.')
else:
    print('  Fits in a Colab session. Proceed.')

del _opt


## Cell 13 — Training loop

In [ ]:
from sklearn.metrics import precision_recall_curve, auc as sklearn_auc

train_hist = dict(total=[], rtt=[], loss_h=[], spike=[], event=[])
val_hist   = dict(total=[], rtt=[], loss_h=[], spike=[], event=[], ev_prauc=[])

for ep in range(start_ep, EPOCHS + 1):
    model.train()
    run = dict(tot=0., r=0., l=0., s=0., e=0., n=0)
    for batch in train_dl:
        xn, xc, yr, yl, msk, spk, ev = [b.to(DEVICE) for b in batch]
        if torch.isnan(xn).any() or torch.isinf(xn).any():
            continue
        optimizer.zero_grad()
        pr, pl, ps, pe = model(xn, xc)
        loss, lr_, ll, ls, le = compute_loss(pr, pl, ps, pe,
                                             yr, yl, msk, spk, ev)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        run['tot'] += loss.item(); run['r'] += lr_; run['l'] += ll
        run['s']   += ls;          run['e'] += le;  run['n'] += 1

    tr = {k: run[k] / max(run['n'], 1) for k in 'tot r l s e'.split()}
    for k, h in zip('tot r l s e'.split(), 'total rtt loss_h spike event'.split()):
        train_hist[h].append(tr[k])

    model.eval()
    vrun = dict(tot=0., r=0., l=0., s=0., e=0., n=0)
    ev_scores, ev_labels_all = [], []

    with torch.no_grad():
        for batch in val_dl:
            xn, xc, yr, yl, msk, spk, ev = [b.to(DEVICE) for b in batch]
            pr, pl, ps, pe = model(xn, xc)
            loss, lr_, ll, ls, le = compute_loss(pr, pl, ps, pe,
                                                 yr, yl, msk, spk, ev)
            vrun['tot'] += loss.item(); vrun['r'] += lr_; vrun['l'] += ll
            vrun['s']   += ls;          vrun['e'] += le;  vrun['n'] += 1
            ev_scores.append(torch.sigmoid(pe).cpu().float().numpy())
            ev_labels_all.append(ev.cpu().numpy())

    vtr = {k: vrun[k] / max(vrun['n'], 1) for k in 'tot r l s e'.split()}
    for k, h in zip('tot r l s e'.split(), 'total rtt loss_h spike event'.split()):
        val_hist[h].append(vtr[k])

    ev_sc = np.concatenate(ev_scores)
    ev_lb = np.concatenate(ev_labels_all).astype(int)
    prec, rec, _ = precision_recall_curve(ev_lb, ev_sc)
    ev_prauc = float(sklearn_auc(rec, prec))
    val_hist['ev_prauc'].append(ev_prauc)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'  Ep {ep:3d} | tr {tr["tot"]:.4f} | '
          f'val {vtr["tot"]:.4f} rtt={vtr["r"]:.4f} '
          f'ev_prauc={ev_prauc:.4f} lr={lr_now:.2e}')

    # ---- checkpointing: ALWAYS before any break -----------------------------
    improved_loss  = vtr['tot'] < best_val
    improved_prauc = ev_prauc > best_event_prauc

    if improved_loss:
        best_val = vtr['tot']
        print(f'  -- new best val_loss={best_val:.4f}')
    if improved_prauc:
        best_event_prauc = ev_prauc
        torch.save(model.state_dict(), ckpt_best_event)
        if ON_COLAB:
            torch.save(model.state_dict(), DRIVE_CKPT_DIR / 'best_event.pt')
        print(f'  -- best_event.pt (ev_prauc={best_event_prauc:.4f})')

    monitored_improved = improved_prauc if EARLY_STOP_ON == 'ev_prauc' \
                                        else improved_loss
    if monitored_improved:
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_best)
        if ON_COLAB:
            torch.save(model.state_dict(), DRIVE_CKPT_DIR / 'best.pt')
        print(f'  -- best.pt (monitor={EARLY_STOP_ON})')
    else:
        patience_ctr += 1

    state_d = dict(model=model.state_dict(), optim=optimizer.state_dict(),
                   sched=scheduler.state_dict(), epoch=ep,
                   best_val=best_val, best_event_prauc=best_event_prauc,
                   patience_ctr=patience_ctr)
    torch.save(state_d, ckpt_last)
    if ON_COLAB:
        torch.save(state_d, DRIVE_CKPT_DIR / 'last.pt')

    if patience_ctr >= PATIENCE:
        print(f'  Early stop at ep {ep} '
              f'({EARLY_STOP_ON} flat for {PATIENCE} epochs)')
        break

print(f'\nBest val total loss : {best_val:.4f}')
print(f'Best event PR-AUC   : {best_event_prauc:.4f}')

## Cell 14 — Loss curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
axes[0].plot(train_hist['total'], label='train')
axes[0].plot(val_hist['total'],   label='val')
axes[0].set_title('Total Loss'); axes[0].legend()
axes[1].plot(train_hist['rtt'],   label='train')
axes[1].plot(val_hist['rtt'],     label='val')
axes[1].set_title('RTT Loss'); axes[1].legend()
axes[2].plot(train_hist['event'], label='train')
axes[2].plot(val_hist['event'],   label='val')
axes[2].set_title('Event Loss'); axes[2].legend()
axes[3].plot(val_hist['ev_prauc'], label='val ev_prauc', color='green')
axes[3].set_title('Event PR-AUC'); axes[3].legend()
plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'loss_curves.png'), dpi=120)
if ON_COLAB:
    plt.savefig(str(DRIVE_CKPT_DIR / 'loss_curves.png'), dpi=120)
plt.show()

## Cell 15 — Evaluation

In [ ]:
from sklearn.metrics import precision_recall_curve, auc as sklearn_auc


def evaluate(model, dl, ckpt_path=None, label=''):
    if ckpt_path and Path(ckpt_path).exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    pred_ms_all, true_ms_all = [], []
    pred_spk_all, true_spk_all = [], []
    pred_ev_all, true_ev_all = [], []
    mask_all = []

    with torch.no_grad():
        for batch in dl:
            xn, xc, yr, yl, msk, spk, ev = [b.to(DEVICE) for b in batch]
            pr, pl, ps, pe = model(xn, xc)
            pred_ms_all.append(np.expm1(pr.cpu().float().numpy().clip(-10, 15)))
            true_ms_all.append(np.expm1(yr.cpu().numpy().clip(-10, 15)))
            pred_spk_all.append(torch.sigmoid(ps).cpu().float().numpy())
            true_spk_all.append(spk.cpu().numpy())
            pred_ev_all.append(torch.sigmoid(pe).cpu().float().numpy())
            true_ev_all.append(ev.cpu().numpy())
            mask_all.append(msk.cpu().numpy())

    pred_ms = np.concatenate(pred_ms_all)
    true_ms = np.concatenate(true_ms_all)
    valid   = np.concatenate(mask_all) == 1
    pv, tv  = pred_ms[valid], true_ms[valid]

    mae  = float(np.mean(np.abs(pv - tv)))
    rmse = float(np.sqrt(np.mean((pv - tv) ** 2)))
    _ok  = tv >= 1.0
    ape  = np.abs(pv[_ok] - tv[_ok]) / tv[_ok] * 100
    mape, medape = float(ape.mean()), float(np.median(ape))

    # MAE restricted to spike bins — the SPIKE_RTT_WEIGHT ablation metric
    spk_flat  = np.concatenate(true_spk_all)[valid].astype(int)
    spk_score = np.concatenate(pred_spk_all)[valid]
    _sm = spk_flat == 1
    mae_spike = float(np.mean(np.abs(pv[_sm] - tv[_sm]))) if _sm.any() else float('nan')
    mae_nonspike = float(np.mean(np.abs(pv[~_sm] - tv[~_sm])))

    print(f'\n=== {label} — RTT Regression ===')
    print(f'  MAE  (all)       : {mae:.2f} ms   (n={valid.sum():,} bins)')
    print(f'  MAE  (spike bins): {mae_spike:.2f} ms   (n={_sm.sum():,}, '
          f'{_sm.mean()*100:.2f}% of valid bins)   <-- ablation metric')
    print(f'  MAE  (non-spike) : {mae_nonspike:.2f} ms')
    print(f'  RMSE : {rmse:.2f} ms')
    print(f'  MAPE : {mape:.2f} %   (median APE {medape:.2f} %, true RTT >= 1 ms)')

    prec, rec, thr = precision_recall_curve(spk_flat, spk_score)
    spk_prauc = float(sklearn_auc(rec, prec))
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
    bi  = int(np.argmax(f1s))
    print(f'\n=== {label} — Per-Bin Spike ===')
    print(f'  base rate: {spk_flat.mean():.4f}')
    print(f'  PR-AUC   : {spk_prauc:.4f}  ({spk_prauc/max(spk_flat.mean(),1e-9):.1f}x base rate)')
    print(f'  best F1  : {f1s[bi]:.4f}  P={prec[bi]:.3f} R={rec[bi]:.3f}')

    ev_true  = np.concatenate(true_ev_all).astype(int)
    ev_score = np.concatenate(pred_ev_all)
    prec_e, rec_e, thr_e = precision_recall_curve(ev_true, ev_score)
    ev_prauc = float(sklearn_auc(rec_e, prec_e))
    f1_e = 2*prec_e[:-1]*rec_e[:-1]/(prec_e[:-1]+rec_e[:-1]+1e-8)
    be   = int(np.argmax(f1_e))
    print(f'\n=== {label} — Sample-Level Event (HO+Cong) ===')
    print(f'  n        : {len(ev_true):,}   n_event: {int(ev_true.sum()):,}')
    print(f'  base rate: {ev_true.mean():.4f}   <-- the only valid reference')
    print(f'  PR-AUC   : {ev_prauc:.4f}  ({ev_prauc/max(ev_true.mean(),1e-9):.1f}x base rate)')
    print(f'  best F1  : {f1_e[be]:.4f}  P={prec_e[be]:.3f} R={rec_e[be]:.3f}')
    print(f'  (best-F1 threshold selected on THIS split — for a reportable '
          f'number, pick the threshold on val and apply it to test.)')

    print(f'\n  {{dataset: combined_MarAprMay | model: v8.1 | split: {label} | '
          f'n={len(ev_true)} n_event={int(ev_true.sum())}}}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(rec, prec, lw=2, label=f'Per-bin spike (AUC={spk_prauc:.3f})')
    axes[0].axhline(spk_flat.mean(), color='grey', ls='--',
                    label=f'Base rate ({spk_flat.mean():.4f})')
    axes[0].scatter([rec[bi]], [prec[bi]], c='red', zorder=5,
                    label=f'Best F1={f1s[bi]:.3f}')
    axes[0].set_title(f'{label} — Per-Bin Spike PR')
    axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
    axes[0].legend(); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

    axes[1].plot(rec_e, prec_e, lw=2, color='green',
                 label=f'Event (AUC={ev_prauc:.3f})')
    axes[1].axhline(ev_true.mean(), color='grey', ls='--',
                    label=f'Base rate ({ev_true.mean():.4f})')
    axes[1].scatter([rec_e[be]], [prec_e[be]], c='red', zorder=5,
                    label=f'Best F1={f1_e[be]:.3f}')
    axes[1].set_title(f'{label} — Event PR')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].legend(); axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

    plt.tight_layout()
    _fn = f'pr_{label.lower().replace(" ", "_")}.png'
    plt.savefig(str(CKPT_DIR / _fn), dpi=120)
    if ON_COLAB:
        plt.savefig(str(DRIVE_CKPT_DIR / _fn), dpi=120)
    plt.show()

    return dict(n=int(len(ev_true)), n_event=int(ev_true.sum()),
                event_base_rate=float(ev_true.mean()),
                MAE_ms=mae, MAE_spike_ms=mae_spike,
                MAE_nonspike_ms=mae_nonspike,
                RMSE_ms=rmse, MAPE_pct=mape, MedAPE_pct=medape,
                spike_prauc=spk_prauc, spike_f1=float(f1s[bi]),
                event_prauc=ev_prauc, event_f1=float(f1_e[be]))


results = {}
for _ck, _tag in ((ckpt_best, 'best'), (ckpt_best_event, 'best_event')):
    print(f'\n=== {_ck.name} ===')
    results[f'{_tag}_val']  = evaluate(model, val_dl,  _ck, f'v8.1 {_tag} val')
    results[f'{_tag}_test'] = evaluate(model, test_dl, _ck, f'v8.1 {_tag} test')

results['config'] = dict(
    spike_rtt_weight=SPIKE_RTT_WEIGHT,
    event_pos_weight=float(EVENT_POS_WEIGHT),
    spike_pos_weight=SPIKE_POS_WEIGHT,
    normal_keep_ratio=NORMAL_KEEP_RATIO,
    early_stop_on=EARLY_STOP_ON,
    sat_vocab_size=int(SAT_VOCAB_SIZE),
)

with open(CKPT_DIR / 'metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
if ON_COLAB:
    with open(DRIVE_CKPT_DIR / 'metrics.json', 'w') as f:
        json.dump(results, f, indent=2)
print('\nmetrics.json saved')

# 6b - embeddings

In [ ]:
CKPT_SUBDIR = 'patchtst_v9.12'

if ON_COLAB:
    LOCAL_ROOT      = Path('/content/rtt_forecast_ds_1s')
    DRIVE_ROOT      = Path('/content/drive/MyDrive/rtt_forecast_ds_1s')
    CKPT_BASE       = LOCAL_ROOT / 'checkpoints'
    DRIVE_CKPT_BASE = DRIVE_ROOT / 'checkpoints'
else:
    LOCAL_ROOT = Path(r'D:\LENS Dataset\combined\rtt_forecast_ds')
    CKPT_BASE  = LOCAL_ROOT / 'checkpoints'

SAMPLE_DIR = '/content/samples'

# Adjust CHECKPOINT path to look in DRIVE_ROOT if ON_COLAB
if ON_COLAB:
    CKPT_DIR = DRIVE_CKPT_BASE / CKPT_SUBDIR
else:
    CKPT_DIR = CKPT_BASE / CKPT_SUBDIR

CHECKPOINT = CKPT_DIR / 'best_event.pt'

# outputs go to the dataset root (Step 8 reads them from DS_ROOT)
OUT_EMB    = LOCAL_ROOT / 'embeddings.csv'
OUT_STAGE1 = LOCAL_ROOT / 'index_stage2.csv'
if ON_COLAB:
    DRIVE_OUT_EMB    = CKPT_BASE / CKPT_SUBDIR / 'embeddings.csv'
    DRIVE_OUT_STAGE1 = CKPT_BASE / CKPT_SUBDIR / 'index_stage2.csv'

#assert ('/content/scalers_v9.8.json').exists(),    'scalers.json not found — wrong root?'
#assert ('/content/index_split_v9_6.csv').exists(), 'index_split_v9_6.csv not found'
assert CHECKPOINT.exists(), (
    f'best_event.pt not found at {CHECKPOINT}. '
    f'Set CKPT_SUBDIR to the folder your v8.1 run actually wrote.')

_mtime = time.strftime('%Y-%m-%d %H:%M', time.localtime(CHECKPOINT.stat().st_mtime))
print('LOCAL_ROOT :', LOCAL_ROOT)
print('CHECKPOINT :', CHECKPOINT)
print('   modified:', _mtime)
print('OUT_EMB    :', OUT_EMB)
print('OUT_STAGE1 :', OUT_STAGE1)

In [ ]:
STEPS_IN, STEPS_OUT, DT_MS = 1500, 100, 10

with open(LOCAL_ROOT / 'feature_schema.json') as f:
    SCHEMA = json.load(f)

STEPS_IN  = int(SCHEMA['steps_in'])      # 1500 for v7 sliding — NOT hardcoded
STEPS_OUT = int(SCHEMA['steps_out'])     # 500
DT_MS     = int(SCHEMA['dt_ms'])         # 10

SELECT_IDX   = SCHEMA['x_num_select_indices']
NUM_FEATURES = SCHEMA['num_features']
F_NUM_BASE   = len(NUM_FEATURES)         # 9
F_NUM        = F_NUM_BASE + 2            # 11 (+ time_since_ho_norm, ho_time_known)

print(f"geometry : steps_in={STEPS_IN} steps_out={STEPS_OUT} dt={DT_MS}ms "
      f"windowing={SCHEMA.get('windowing')}")

# ── patch size: must divide STEPS_IN and make the decoder emit exactly STEPS_OUT
def pick_patch_size(steps_in, steps_out, prefer_n_patches=50):
    ok = [p for p in range(2, steps_in + 1)
          if steps_in % p == 0 and steps_out % (steps_in // p) == 0]
    if not ok:
        raise ValueError(f'no valid patch_size for {steps_in}/{steps_out}')
    return min(ok, key=lambda p: abs(steps_in // p - prefer_n_patches))

PATCH_SIZE      = pick_patch_size(STEPS_IN, STEPS_OUT, prefer_n_patches=50)
N_PATCHES       = STEPS_IN // PATCH_SIZE
STEPS_PER_TOKEN = STEPS_OUT // N_PATCHES
assert N_PATCHES * STEPS_PER_TOKEN == STEPS_OUT
print(f"patching : patch_size={PATCH_SIZE} n_patches={N_PATCHES} "
      f"steps_per_token={STEPS_PER_TOKEN} -> emits {N_PATCHES*STEPS_PER_TOKEN} == {STEPS_OUT} OK")

# ── scalers: verify by FINGERPRINT, not by the old hardcoded mean==80.2735.
# The sliding data was refit (mean≈79.37) so the old assert would WRONGLY reject it.
with open(target / 'scalers.json') as f:
    _RAW_SCALERS = json.load(f)

_FP = _RAW_SCALERS.get('__fingerprint__')
if _FP is None:
    raise RuntimeError("scalers.json has no __fingerprint__ — re-run step5_fit_scalers.py")
_mismatch = [f"{k}: scalers={_FP.get(k)!r} vs schema={SCHEMA.get(k)!r}"
             for k in ('steps_in', 'steps_out', 'window_s', 'step_s')
             if _FP.get(k) != SCHEMA.get(k)]
if _mismatch:
    raise RuntimeError("STALE SCALERS — fingerprint mismatch:\n  " + "\n  ".join(_mismatch))
if list(_FP.get('num_features', [])) != list(NUM_FEATURES):
    raise RuntimeError("Scalers fitted on a different feature list.")

SCALERS = {k: v for k, v in _RAW_SCALERS.items() if k in NUM_FEATURES}
assert set(NUM_FEATURES) == set(SCALERS), f"Missing: {set(NUM_FEATURES) - set(SCALERS)}"
_p = SCALERS['rtt_ms']['params']
print(f"scalers  : fingerprint OK  rtt_ms clip={_p['clip_max']:.0f} "
      f"mean={_p['mean']:.4f} scale={_p['scale']:.4f} (refit on sliding data)")

CAT_COL_IDX = 0
_idx = pd.read_csv(LOCAL_ROOT / 'index_split_roccet_05s.csv')
assert 'event_id' in _idx.columns, "index_split_roccet_05s.csv has no event_id — needed for per-event eval"
print(f"index    : {len(_idx):,} windows, event_id present OK")

In [ ]:
_N_SCAN = 4000
_rng_v  = np.random.default_rng(0)
_scan_i = _rng_v.choice(len(_idx), size=min(_N_SCAN, len(_idx)), replace=False)
_seen = set()
for _j in _scan_i:
    _d = np.load(Path(SAMPLE_DIR) / _idx['sample'].iloc[int(_j)], allow_pickle=False)
    _seen.update(np.unique(_d['X_cat'][:, CAT_COL_IDX]).tolist())
_scan_max = int(max(_seen))
SAT_VOCAB_SIZE = _scan_max + 200          # matches v8.1 (600 -> 800)
CAT_VOCAB_SIZES = [SAT_VOCAB_SIZE]
print(f'scan max satellite id = {_scan_max}  ->  SAT_VOCAB_SIZE = {SAT_VOCAB_SIZE}')
assert SAT_VOCAB_SIZE == 800, (
    f'SAT_VOCAB_SIZE={SAT_VOCAB_SIZE} but v8.1 trained with 800. '
    f'nn.Embedding will not match the checkpoint. Investigate before proceeding.')

SAT_CHANGE_IDX = NUM_FEATURES.index('sat_change')     # 8
SINR_IDX       = NUM_FEATURES.index('AVERAGE_SINR')   # 3
SINR_DELTA_IDX = NUM_FEATURES.index('sinr_delta')     # 4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
print(f'NUM_FEATURES : {NUM_FEATURES}')
print(f'F_NUM={F_NUM}  samples={len(_idx):,}  device={DEVICE}')

In [ ]:
FFILL_COLS = ('rtt_ms',)


def _ffill_1d(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32, copy=True)
    fin = np.isfinite(a)
    if not fin.any():
        return a
    idx = np.where(fin, np.arange(len(a)), 0)
    np.maximum.accumulate(idx, out=idx)
    out = a[idx]
    first = int(np.argmax(fin))
    if first > 0:
        out[:first] = a[first]
    return out


def apply_scalers(X_num: np.ndarray) -> np.ndarray:
    out = X_num.copy().astype(np.float32)
    for i, col in enumerate(NUM_FEATURES):
        if col in FFILL_COLS:
            out[:, i] = _ffill_1d(out[:, i])
    for i, col in enumerate(NUM_FEATURES):
        if col not in SCALERS:
            continue
        info, p = SCALERS[col], SCALERS[col]['params']
        valid = np.isfinite(out[:, i])
        if not valid.any():
            continue
        t = info['type']
        if t == 'clipped_standard':
            v = np.clip(out[valid, i], 0.0, p['clip_max'])
            out[valid, i] = (v - p['mean']) / p['scale']
        elif t == 'standard':
            out[valid, i] = (out[valid, i] - p['mean']) / p['scale']
        elif t == 'log1p_standard':
            v = np.clip(out[valid, i], 0, p['clip_max'])
            out[valid, i] = (np.log1p(v) - p['mean']) / p['scale']
        elif t == 'robust':
            out[valid, i] = (out[valid, i] - p['center']) / p['scale']
    return np.nan_to_num(out, nan=0.0)


print('apply_scalers (with ffill) defined OK')

In [ ]:
SINR_DELTA_WINDOW = 100        # 100 bins x 10 ms = 1 s (matches v8.1)


class EmbeddingDataset(Dataset):
    def __init__(self, index_df: pd.DataFrame):
        self.samples = index_df['sample'].tolist()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = np.load(Path(SAMPLE_DIR) / self.samples[idx], allow_pickle=False)
        X_num_raw = d['X_num'][:, SELECT_IDX].copy()      # (2500, 9)

        base_s   = float(d['time_since_ho'])
        ho_known = float(d['ho_time_known'])
        sat_col  = X_num_raw[:, SAT_CHANGE_IDX]
        sat_bins = np.where(sat_col > 0.5)[0]
        indices  = np.arange(STEPS_IN, dtype=np.float64)

        if len(sat_bins) == 0:
            tsb = (base_s + (indices + 1) * (DT_MS / 1000.0)).astype(np.float32)
            tsn = np.clip(tsb / 150.0, 0.0, 2.0)
            hkc = np.full(STEPS_IN, ho_known, dtype=np.float32)
        else:
            counts    = np.searchsorted(sat_bins, np.arange(STEPS_IN), side='right')
            has_prior = counts > 0
            last_ho   = sat_bins[np.clip(counts - 1, 0, len(sat_bins) - 1)]
            tsb = np.where(
                has_prior,
                (indices - last_ho) * (DT_MS / 1000.0),
                base_s + (indices + 1) * (DT_MS / 1000.0),
            ).astype(np.float32)
            tsn = np.clip(tsb / 150.0, 0.0, 2.0)
            hkc = np.where(has_prior, 1.0, ho_known).astype(np.float32)

        extra   = np.stack([tsn, hkc], axis=1)
        _scaled = apply_scalers(X_num_raw)

        W = SINR_DELTA_WINDOW
        _scaled[:, SINR_DELTA_IDX] = np.concatenate([
            np.zeros(W, np.float32),
            _scaled[W:, SINR_IDX] - _scaled[:-W, SINR_IDX],
        ]).astype(np.float32)

        X_num = np.concatenate([_scaled, extra], axis=1)  # (2500, 11)
        X_cat = d['X_cat'][:, CAT_COL_IDX:CAT_COL_IDX+1].astype(np.int64)
        X_cat = np.clip(X_cat, 0, SAT_VOCAB_SIZE)

        return (torch.from_numpy(X_num), torch.from_numpy(X_cat),
                self.samples[idx])


print('EmbeddingDataset defined OK')

In [ ]:
class PatchTSTForecaster(nn.Module):
    def __init__(self, f_num=11, cat_vocab_sizes=None,
                 emb_dim=8, patch_size=None,
                 d_model=256, n_heads=8, n_layers=4, d_ff=512,
                 dropout=0.1, spike_dropout=0.1):
        super().__init__()
        if patch_size is None:
          patch_size = PATCH_SIZE
        if cat_vocab_sizes is None:
            cat_vocab_sizes = []
        self.patch_size      = patch_size
        self.n_patches       = STEPS_IN // patch_size
        self.steps_per_token = STEPS_OUT // self.n_patches
        self.embeddings = nn.ModuleList([
            nn.Embedding(vs + 1, emb_dim, padding_idx=0)
            for vs in cat_vocab_sizes])
        f_total        = f_num + emb_dim * len(cat_vocab_sizes)
        self.input_pad = (-f_total) % 8
        f_total_padded = f_total + self.input_pad
        small_dim      = 32
        self.ts_proj   = nn.Sequential(nn.Linear(f_total_padded, small_dim), nn.GELU())
        patch_dim      = patch_size * small_dim
        self.patch_embed = nn.Sequential(
            nn.Linear(patch_dim, d_model), nn.LayerNorm(d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.drop = nn.Dropout(dropout)
        self.rtt_decoder = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1))
        self.loss_decoder = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1))
        self.spike_decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(spike_dropout),
            nn.Linear(d_model // 2, 1))
        self.event_head = nn.Sequential(
            nn.Linear(d_model, d_model // 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1))
        assert self.n_patches * self.patch_size == STEPS_IN
        assert self.n_patches * self.steps_per_token == STEPS_OUT

    def encode(self, X_num, X_cat):
        """Everything up to the pooled (B, d_model) representation."""
        B = X_num.shape[0]
        if self.embeddings:
            emb = torch.cat(
                [e(X_cat[:, :, i].clamp(0, e.num_embeddings - 1))
                 for i, e in enumerate(self.embeddings)], dim=-1)
            x = torch.cat([X_num, emb], dim=-1)
        else:
            x = X_num
        if self.input_pad > 0:
            x = F.pad(x, (0, self.input_pad))
        x = self.ts_proj(x)
        x = x.reshape(B, self.n_patches, self.patch_size * x.shape[-1])
        x = self.patch_embed(x) + self.pos_embed
        x = self.drop(x)
        x = self.transformer(x)          # (B, n_patches, d_model)
        return x

    def forward(self, X_num, X_cat):
        x = self.encode(X_num, X_cat)
        pooled = x.mean(dim=1)                              # (B, d_model)
        pred_event = self.event_head(pooled).squeeze(-1)
        xr = self.drop(x.repeat_interleave(self.steps_per_token, dim=1))
        pred_rtt   = self.rtt_decoder(xr).squeeze(-1)
        pred_loss  = self.loss_decoder(xr).squeeze(-1)
        pred_spike = self.spike_decoder(xr).squeeze(-1)
        return pred_rtt, pred_loss, pred_spike, pred_event, pooled


print('PatchTSTForecaster defined OK')

In [ ]:
model = PatchTSTForecaster(
    f_num=F_NUM, cat_vocab_sizes=CAT_VOCAB_SIZES,
    emb_dim=8, patch_size=PATCH_SIZE,
    d_model=256, n_heads=8, n_layers=4, d_ff=512,
    dropout=0.1, spike_dropout=0.3)

state = torch.load(CHECKPOINT, map_location=DEVICE)
if isinstance(state, dict) and 'model' in state:
    state = state['model']
missing, unexpected = model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()
print('Checkpoint loaded (strict=True — all keys matched).')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

idx = pd.read_csv(target / 'index_split_roccet_05s.csv')
assert idx['sample'].nunique() == len(idx), 'duplicate sample keys'
print(f'Samples to process: {len(idx):,}  '
      f"splits={idx['split'].value_counts().to_dict()}")

In [ ]:
loader = DataLoader(EmbeddingDataset(idx), batch_size=128, shuffle=False,
                    num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

emb_rows, s1_rows, names_all = [], [], []
_bin_t = np.arange(STEPS_OUT, dtype=np.float32)
_bin_t = (_bin_t - _bin_t.mean())        # centred, for a cheap slope estimate
_den   = float((_bin_t ** 2).sum())

t0 = time.time()
with torch.no_grad():
    for bi, (xn, xc, names) in enumerate(loader):
        xn, xc = xn.to(DEVICE), xc.to(DEVICE)
        x = model.encode(xn, xc)                 # (B, n_patches, d_model)
        pooled = x.mean(dim=1)                    # (B, 256) — the embedding

        pred_event = model.event_head(pooled).squeeze(-1)
        xr = x.repeat_interleave(model.steps_per_token, dim=1)
        pred_rtt   = model.rtt_decoder(xr).squeeze(-1)     # (B, 500) log-space
        pred_spike = model.spike_decoder(xr).squeeze(-1)   # (B, 500) logits

        ev_prob   = torch.sigmoid(pred_event).cpu().numpy()
        spk_prob  = torch.sigmoid(pred_spike).cpu().numpy()      # (B, 500)
        rtt_ms    = np.expm1(pred_rtt.cpu().float().numpy().clip(-10, 15))  # (B,500)
        emb_np    = pooled.cpu().float().numpy()

        # per-sample Stage-1 summaries (what Step 8 gates / features on)
        r_mean  = rtt_ms.mean(1)
        r_max   = rtt_ms.max(1)
        r_std   = rtt_ms.std(1)
        r_slope = (rtt_ms * _bin_t).sum(1) / _den           # ms per bin
        s_max   = spk_prob.max(1)
        s_mean  = spk_prob.mean(1)

        for k, nm in enumerate(names):
            names_all.append(nm)
            emb_rows.append(emb_np[k])
            s1_rows.append((ev_prob[k], s_max[k], s_mean[k],
                            r_mean[k], r_max[k], r_slope[k], r_std[k]))

        if bi % 50 == 0:
            done = len(names_all)
            print(f'  batch {bi:4d}  {done:>7,}/{len(idx):,}  '
                  f'{done/(time.time()-t0):.0f} samp/s')

print(f'Extracted {len(names_all):,} in {(time.time()-t0)/60:.1f} min')

In [ ]:
emb_arr = np.asarray(emb_rows, dtype=np.float32)
emb_cols = [f'emb_{i:03d}' for i in range(emb_arr.shape[1])]
emb_df = pd.DataFrame(emb_arr, columns=emb_cols)
emb_df.insert(0, 'sample', names_all)

s1_df = pd.DataFrame(s1_rows, columns=[
    'event_prob', 'spike_prob_max', 'spike_prob_mean',
    'pred_rtt_mean', 'pred_rtt_max', 'pred_rtt_slope', 'pred_rtt_std'])
s1_df.insert(0, 'sample', names_all)

# join split/event_class/event_id back on so Step 8 can gate, validate, and
# aggregate windows -> events for the per-event congestion F1 (the headline metric)
_carry = ['sample', 'split', 'event_class']
if 'event_id' in idx.columns:
    _carry.append('event_id')
else:
    print("WARNING: no event_id in index — per-event evaluation will NOT be possible")
s1_df = s1_df.merge(idx[_carry], on='sample', how='left')

# --- sanity: event_prob must separate true events from normals -------------
_m = s1_df['event_class'].isin([1, 2])
print(f"embedding dim: {emb_arr.shape[1]}  (expect 256)")
print(f"event_prob  mean on TRUE events : {s1_df.loc[_m,'event_prob'].mean():.3f}")
print(f"event_prob  mean on TRUE normal : {s1_df.loc[~_m,'event_prob'].mean():.3f}")
print("  (events should be clearly higher; if not, wrong checkpoint/preproc)")
assert emb_arr.shape[1] == 256, f'embedding dim {emb_arr.shape[1]} != 256'
assert not emb_df[emb_cols].isna().any().any(), 'NaNs in embeddings'

# confirm event_id came through (needed downstream)
if 'event_id' in s1_df.columns:
    print(f"event_id carried through: {s1_df['event_id'].notna().sum():,} rows have it")

DRIVE_OUT_EMB    = DRIVE_CKPT_BASE / CKPT_SUBDIR / 'embeddings.csv'
DRIVE_OUT_STAGE1 = DRIVE_CKPT_BASE / CKPT_SUBDIR / 'index_stage2.csv'
emb_df.to_csv(DRIVE_OUT_EMB, index=False)
s1_df.to_csv(DRIVE_OUT_STAGE1, index=False)
print(f'saved {OUT_EMB}  ({emb_df.shape})')
print(f'saved {OUT_STAGE1}  ({s1_df.shape})')

In [ ]:
# after the to_csv calls, force Drive to flush:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')   # remount

In [ ]:
CKPT_SUBDIR = 'patchtst_v9.12'
DRIVE_OUT_EMB    = DRIVE_CKPT_BASE / CKPT_SUBDIR / 'embeddings.csv'
DRIVE_OUT_STAGE1 = DRIVE_CKPT_BASE / CKPT_SUBDIR / 'index_stage2.csv'
emb_df.to_csv(DRIVE_OUT_EMB, index=False)
s1_df.to_csv(DRIVE_OUT_STAGE1, index=False)
print(f'backed up both to {DRIVE_ROOT}')
# after the to_csv calls, force Drive to flush:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')   # remount